# 03 - Exploratory Data Analysis

Phase 3 answers the five research questions using the cleaned primary dataset and a LinkedIn validation sample.

Note: the primary source provides `posted_year`, not a true posting date, so the trend analysis is annual rather than monthly. That is an honest constraint of the raw data.

## What this notebook covers

1. Skill frequency and category mix
2. Annual trend analysis and top rising/declining skills
3. Skill co-occurrence matrix
4. Skill demand by experience level
5. GenAI validation check using the LinkedIn sample

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src.phase3_eda import (
    annual_skill_trends,
    build_phase3_artifacts,
    cooccurrence_matrix,
    experience_level_skill_crosstab,
    genai_validation,
    load_linkedin_validation,
    load_primary_clean,
    plot_annual_trends,
    plot_cooccurrence_heatmap,
    plot_experience_share_heatmap,
    plot_genai_validation,
    plot_skill_category_bar,
    plot_top_skills_bar,
    skill_category_frequency,
    top_skill_frequency,
)

sns.set_theme(style='whitegrid')

In [ ]:
primary = load_primary_clean()
linkedin_validation = load_linkedin_validation()

print('Primary shape:', primary.shape)
print('LinkedIn validation shape:', linkedin_validation.shape)
print('\nPrimary summary:')
display(primary[['posted_year', 'skill', 'skill_category', 'experience_level']].head())

## RQ1 - Which skills appear most frequently across all postings?

The primary dataset is highly concentrated: the 11 tracked skills are all frequent, with a small edge for cloud and ML platforms.

In [ ]:
top_skills = top_skill_frequency(primary)
category_counts = skill_category_frequency(primary)

display(top_skills)
display(category_counts)

fig1 = plot_top_skills_bar(top_skills)
plt.show()

fig2 = plot_skill_category_bar(category_counts)
plt.show()

## RQ2 - Which skills are rising vs. declining over time?

Because the raw source only gives annual postings, this answer uses yearly share of skill mentions instead of monthly counts. The shifts are modest, which suggests the market is broadly stable rather than highly volatile.

In [ ]:
annual, rising, declining = annual_skill_trends(primary)

display(rising)
display(declining)

fig3 = plot_annual_trends(annual)
fig3.show()

## RQ3 - Which skills co-occur most often in the same job posting?

The co-occurrence matrix shows that the main skills are often requested together, which fits a stacked job requirement pattern rather than isolated single-skill roles.

In [ ]:
matrix, pairs = cooccurrence_matrix(primary)

display(pairs.head(20))
fig4 = plot_cooccurrence_heatmap(matrix)
plt.show()

## RQ4 - How does skill demand differ by experience level?

Experience-level differences are present, but they are subtle because the dataset is balanced. Senior roles lean slightly more toward Azure, GCP, and PyTorch, while entry and mid roles stay close to the overall mix.

In [ ]:
experience_counts, experience_shares = experience_level_skill_crosstab(primary)

display(experience_counts)
display(experience_shares)

fig5 = plot_experience_share_heatmap(experience_shares)
plt.show()

## RQ5 - Are GenAI skills a statistically significant post-2023 trend?

The primary dataset does not contain GenAI terms at all, so the best available validation is the LinkedIn sample. This is not a literal 2022-vs-2024 comparison; it is a post-2023 signal check using the available secondary source.

In [ ]:
validation_rows, genai_summary, chi2, p_value, dof, expected = genai_validation(linkedin_validation)

print(f'Chi-square: {chi2:.3f}')
print(f'p-value: {p_value:.6f}')
print(f'dof: {dof}')
display(genai_summary)

fig6 = plot_genai_validation(genai_summary)
plt.show(fig6)

## Phase 3 takeaway

The primary dataset says the market is broad and stable: the same core skills recur every year, and the rank order shifts only slightly. The LinkedIn sample adds a useful validation signal for GenAI, where LLM and RAG appear enough to support a significant difference by job level.

This sets up Phase 4 well, because we now know which results deserve publication-ready charts and which ones should be framed as validation or caveats.